# Colab + Gradio: fixed v7

Виправлено помилку `NameError: ui_segmentation is not defined`.
У цій версії всі функції `ui_segmentation`, `ui_bertweet`, `ui_profile` визначені явно.

In [ ]:
!pip -q install -U gradio sentence-transformers umap-learn hdbscan plotly scipy wordcloud\
    transformers datasets accelerate scikit-learn matplotlib emoji==0.6.0

import os
import re
import math
import random
import warnings
from typing import List, Tuple

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sentence_transformers import SentenceTransformer
import umap
import hdbscan

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_distances
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform
from plotly.subplots import make_subplots
from wordcloud import WordCloud

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed
)

import gradio as gr

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 200)

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

APP_STATE = {
    "trained_model": None,
    "trained_tokenizer": None,
    "segmentation_df": None,
    "segment_summary": None,
    "text_col": None,
    "user_col": None,
    "selected_user": None,
    "evaluated_segments": None,
}




MENTION_RE = re.compile(r"@\w+")
RT_RE = re.compile(r"^RT\s+@\w+:\s*", flags=re.IGNORECASE)
URL_RE = re.compile(r"https?://\S+|www\.\S+")
NONWORD_RE = re.compile(r"[^a-zA-ZА-Яа-яІіЇїЄєҐґ#\s'-]")
MULTISPACE_RE = re.compile(r"\s+")

UK_STOP = {
    "і","й","та","або","але","а","у","в","на","до","з","із","зі","для","про","під","над","при","по",
    "це","цей","ця","ці","те","той","такий","таке","такі","що","щоб","як","коли","де","який","яка",
    "які","яке","не","ні","так","то","ж","же","би","б","було","бути","є","був","була","були","ми",
    "ви","вони","він","вона","воно","мене","тобі","собі","його","її","їх","наш","ваш","свій","свої",
    "дуже","ще","вже","лише","тільки","після","перед","через","тому","тут","там","ось"
}

EN_STOP = {
    "the","a","an","to","of","and","or","in","on","for","with","at","from","as","is","are","was","were",
    "it","this","that","these","those","i","you","he","she","we","they","me","my","your","our","their",
    "be","been","being","do","did","does","not","no","yes","so","just","im","i'm","dont","don't","cant","can't",
    "very","too","have","has","had","will","would","can","could","should","about","into","than","then","there",
    "here","also","if","because","while","when","where","who","whom","which","what","why","how",
    "his","her","hers","him","them","their","ours","mine","yours","itself","himself","herself",
    "one","two","three","really","still","much","many","more","most","some","any","every",
    "thing","things","stuff","someone","anyone","everyone","something","anything","everything",
    "say","says","said","told","tell","tells","make","made","let","lets","using","used","use",
    "go","going","went","come","coming","came","take","took","taken","look","looks","looked",
    "like","liked","haha","lol","yeah","okay","ok","nah","omg","amp","user","users","rt"
}

SOCIAL_MEDIA_NOISE = {
    "user","users","rt","amp","im","ive","id","ill","dont","didnt","doesnt","cant","couldnt",
    "twitter","tweet","tweets","reddit","post","posts","comment","comments",
    "thing","things","stuff","someone","anyone","everyone","anything","something",
    "guy","guys","girl","girls","boy","boys","man","men","woman","women",
    "people","person","said","says","tell","told","let","lets","using","used","one",
    "really","still","much","many","also","well","yeah","okay","ok","nah","haha","lol",
    "ive","youre","theyre","hes","shes","thats","theres","whats"
}

def stopwords_for_lang(lang="uk"):
    if lang == "uk":
        return sorted(list(UK_STOP | EN_STOP | SOCIAL_MEDIA_NOISE))
    return sorted(list(EN_STOP | SOCIAL_MEDIA_NOISE))

def normalize_for_topics(text: str) -> str:
    t = str(text)
    t = RT_RE.sub("", t)
    t = URL_RE.sub(" ", t)
    t = MENTION_RE.sub(" ", t)
    t = t.replace("&amp;", " ")
    t = NONWORD_RE.sub(" ", t)
    t = re.sub(r"\b\d+\b", " ", t)
    t = MULTISPACE_RE.sub(" ", t).strip().lower()
    return t

def clean_keywords_string(s: str) -> str:
    s = str(s)
    s = re.sub(r"\b\d+\b", " ", s)
    s = re.sub(r"\s+", " ", s).strip(" ,;")
    return s

def token_is_bad(tok: str) -> bool:
    tok = tok.strip().lower()
    if len(tok) < 3:
        return True
    if tok in SOCIAL_MEDIA_NOISE or tok in EN_STOP or tok in UK_STOP:
        return True
    if re.search(r"\d", tok):
        return True
    return False

def term_is_bad(term: str) -> bool:
    term = clean_keywords_string(term.lower())
    if not term:
        return True

    parts = [p for p in term.split() if p.strip()]
    if len(parts) == 0:
        return True

    if all(token_is_bad(p) for p in parts):
        return True

    if any(p in SOCIAL_MEDIA_NOISE or p in EN_STOP or p in UK_STOP for p in parts):
        return True

    if len(set(parts)) < len(parts):
        return True

    banned_fragments = {
        "user user", "one day", "right now", "years ago", "feel like", "look like",
        "said user", "user said", "people say", "something like"
    }
    if term in banned_fragments:
        return True

    return False

def postprocess_terms(terms, max_terms=8):
    cleaned = []
    seen = set()

    for term in terms:
        term = clean_keywords_string(term)
        if term_is_bad(term):
            continue

        parts = term.split()
        if any(token_is_bad(p) for p in parts):
            continue

        if term in seen:
            continue

        cleaned.append(term)
        seen.add(term)

        if len(cleaned) >= max_terms:
            break

    return cleaned




def extract_keywords_ctfidf(
    texts: List[str],
    labels: List[int],
    top_n: int = 24,
    final_top_n: int = 8,
    min_df: int = 2,
    ngram_range: Tuple[int, int] = (1, 3),
    lang: str = "uk"
) -> pd.DataFrame:
    tmp = pd.DataFrame({"text": texts, "cluster": labels})
    tmp = tmp[tmp["cluster"] != -1].copy()

    if tmp.empty:
        return pd.DataFrame(columns=["cluster", "keywords"])

    agg = tmp.groupby("cluster")["text"].apply(
        lambda x: " ".join(map(normalize_for_topics, x))
    ).reset_index()

    vect = CountVectorizer(
        stop_words=stopwords_for_lang(lang),
        min_df=min_df,
        ngram_range=ngram_range,
        token_pattern=r"(?u)\b[a-zA-ZА-Яа-яІіЇїЄєҐґ][a-zA-ZА-Яа-яІіЇїЄєҐґ'-]+\b"
    )

    X = vect.fit_transform(agg["text"])
    tf = X.toarray().astype(float)
    tf = tf / np.clip(tf.sum(axis=1, keepdims=True), 1e-9, None)
    df_term = (X > 0).sum(axis=0).A1
    idf = np.log((1 + X.shape[0]) / (1 + df_term)) + 1
    ctfidf = tf * idf[None, :]
    vocab = np.array(vect.get_feature_names_out())

    rows = []
    for i, cl in enumerate(agg["cluster"].tolist()):
        idx = np.argsort(-ctfidf[i])[:top_n]
        raw_terms = vocab[idx].tolist()
        clean_terms = postprocess_terms(raw_terms, max_terms=final_top_n)
        rows.append({
            "cluster": int(cl),
            "keywords": ", ".join(clean_terms)
        })

    return pd.DataFrame(rows)

def assign_segment_names(summary_df: pd.DataFrame, max_terms_in_name: int = 2) -> pd.DataFrame:
    out = summary_df.copy()
    out["keywords"] = out["keywords"].fillna("").map(clean_keywords_string)

    names = []
    for _, row in out.iterrows():
        kws = [p.strip() for p in str(row["keywords"]).split(",") if p.strip()]
        kws = [k for k in kws if not term_is_bad(k)]

        if len(kws) >= max_terms_in_name:
            base = " / ".join(kws[:max_terms_in_name])
        elif len(kws) == 1:
            base = kws[0]
        else:
            base = "невизначена тема"

        final_name = f"{base} [{int(row['cluster'])}]"
        names.append(final_name)

    out["segment_name"] = names
    return out

def cluster_examples(df_user, cluster_col="cluster", text_col="post_text", n_examples=3):
    rows = []
    for cl, sub in df_user[df_user[cluster_col] != -1].groupby(cluster_col):
        seg_name = sub["segment_name"].iloc[0] if "segment_name" in sub.columns else f"Сегмент {cl}"
        for t in sub[text_col].head(n_examples).tolist():
            rows.append({"cluster": int(cl), "segment_name": seg_name, "example": t})
    return pd.DataFrame(rows)

def plot_clusters_clean(viz: pd.DataFrame, noise_label: int = -1):
    fig = go.Figure()
    for cl in sorted(viz["cluster"].unique()):
        sub = viz[viz["cluster"] == cl]
        name = "Шум" if cl == noise_label else f"Сегмент {cl}"
        fig.add_trace(go.Scattergl(
            x=sub["x"], y=sub["y"],
            mode="markers",
            name=name,
            text=sub["text_short"],
            marker=dict(size=7, opacity=0.78)
        ))
    fig.update_layout(
        title="UMAP-представлення комунікативних сегментів",
        xaxis_title="UMAP-1",
        yaxis_title="UMAP-2",
        template="plotly_white",
        height=560,
        margin=dict(l=30, r=20, t=50, b=30)
    )
    return fig

def centroid_distance_heatmap(embeddings: np.ndarray, labels: np.ndarray):
    uniq = sorted([x for x in np.unique(labels) if x != -1])
    if len(uniq) < 2:
        fig = go.Figure()
        fig.update_layout(template="plotly_white", title="Недостатньо сегментів для теплової карти", height=420)
        return fig

    centroids = []
    for cl in uniq:
        centroids.append(embeddings[labels == cl].mean(axis=0))
    centroids = np.vstack(centroids)

    dist = cosine_distances(centroids)
    condensed = squareform(dist, checks=False)
    Z = linkage(condensed, method="average")
    order = leaves_list(Z)

    ordered_labels = [uniq[i] for i in order]
    ordered_dist = dist[np.ix_(order, order)]

    fig = px.imshow(
        ordered_dist,
        x=[f"Сегмент {c}" for c in ordered_labels],
        y=[f"Сегмент {c}" for c in ordered_labels],
        color_continuous_scale="Blues",
        title="Косинусні відстані між центроїдами сегментів"
    )
    fig.update_layout(template="plotly_white", height=480, margin=dict(l=30, r=30, t=60, b=30))
    return fig

def run_segmentation(
    csv_path: str,
    text_col: str,
    user_col: str,
    target_user: str,
    model_name: str,
    min_cluster_size: int,
    min_samples: int,
    top_n_keywords: int,
    lang: str = "uk",
    max_segments_display: int = 15,
):
    df = pd.read_csv(csv_path)
    df = df[[user_col, text_col]].copy()
    df[text_col] = df[text_col].astype(str).fillna("").str.strip()
    df = df[df[text_col].str.len() > 0].reset_index(drop=True)

    if target_user == "__AUTO__":
        user_stats = df.groupby(user_col).size().reset_index(name="n_posts").sort_values("n_posts", ascending=False)
        target_user_val = user_stats.iloc[0][user_col]
    else:
        target_user_val = pd.Series([target_user]).astype(df[user_col].dtype)[0]

    user_df = df[df[user_col] == target_user_val].copy().reset_index(drop=True)
    if len(user_df) < 10:
        raise ValueError("Для вибраного автора замало повідомлень для змістовної сегментації.")

    texts = user_df[text_col].tolist()
    model = SentenceTransformer(model_name)
    emb = model.encode(texts, show_progress_bar=False, normalize_embeddings=True)

    reducer10 = umap.UMAP(
        n_components=10,
        metric="cosine",
        random_state=RANDOM_STATE,
        n_neighbors=min(15, max(5, len(texts)//10))
    )
    emb10 = reducer10.fit_transform(emb)

    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=True
    )
    labels = clusterer.fit_predict(emb10)
    probs = getattr(clusterer, "probabilities_", np.ones(len(labels)))

    reducer2 = umap.UMAP(n_components=2, metric="cosine", random_state=RANDOM_STATE)
    emb2 = reducer2.fit_transform(emb)

    viz = pd.DataFrame({
        "x": emb2[:, 0],
        "y": emb2[:, 1],
        "cluster": labels,
        "prob": probs,
        "text_short": [t[:160] + ("..." if len(t) > 160 else "") for t in texts]
    })

    user_df["cluster"] = labels
    user_df["cluster_prob"] = probs

    keywords = extract_keywords_ctfidf(
        texts,
        labels,
        top_n=max(24, top_n_keywords * 3),
        final_top_n=top_n_keywords,
        lang=lang
    )

    summary = (
        user_df.groupby("cluster")
        .agg(
            n_messages=(text_col, "size"),
            mean_cluster_prob=("cluster_prob", "mean")
        )
        .reset_index()
        .sort_values(["cluster"])
    )
    summary = summary.merge(keywords, on="cluster", how="left")
    summary = assign_segment_names(summary)

    summary_display = (
        summary[summary["cluster"] != -1]
        .sort_values(["n_messages", "mean_cluster_prob"], ascending=[False, False])
        .head(max_segments_display)
        .copy()
    )
    allowed_clusters = set(summary_display["cluster"].tolist())

    user_df = user_df.merge(summary[["cluster", "segment_name", "keywords"]], on="cluster", how="left")
    user_df_display = user_df[user_df["cluster"].isin(allowed_clusters)].copy()

    examples = cluster_examples(user_df_display, cluster_col="cluster", text_col=text_col, n_examples=3)

    noise_share = float((labels == -1).mean())
    n_clusters = int(len(set(labels)) - (1 if -1 in labels else 0))

    metrics_md = f"""
**Вибраний автор:** `{target_user_val}`
**Кількість повідомлень:** `{len(user_df)}`
**Кількість виявлених сегментів:** `{n_clusters}`
**Частка шуму:** `{noise_share:.3f}`
**Показано сегментів:** `{len(summary_display)}`
"""

    fig_scatter = plot_clusters_clean(viz)
    fig_dist = centroid_distance_heatmap(emb, labels)

    APP_STATE["segmentation_df"] = user_df_display.copy()
    APP_STATE["segment_summary"] = summary_display.copy()
    APP_STATE["text_col"] = text_col
    APP_STATE["user_col"] = user_col
    APP_STATE["selected_user"] = target_user_val

    return metrics_md, fig_scatter, fig_dist, summary_display, examples




def clean_binary_df(df, text_col, label_col):
    df = df[[text_col, label_col]].copy()
    df[text_col] = df[text_col].astype(str).str.strip()
    df = df[df[text_col].notna() & (df[text_col].str.len() > 0)]
    df[label_col] = pd.to_numeric(df[label_col], errors="coerce")
    df = df[df[label_col].notna()]
    df[label_col] = df[label_col].astype(int)
    df = df[df[label_col].isin([0, 1])].reset_index(drop=True)
    return df

def balance_train_df(train_df, label_col, mode="upsample"):
    vc = train_df[label_col].value_counts()
    if len(vc) < 2:
        return train_df.copy()
    maj = vc.idxmax()
    minc = vc.idxmin()
    df_maj = train_df[train_df[label_col] == maj]
    df_min = train_df[train_df[label_col] == minc]
    if mode == "downsample":
        df_maj = df_maj.sample(n=len(df_min), random_state=RANDOM_STATE)
        out = pd.concat([df_maj, df_min], axis=0)
    else:
        df_min = df_min.sample(n=len(df_maj), replace=True, random_state=RANDOM_STATE)
        out = pd.concat([df_maj, df_min], axis=0)
    return out.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

def compute_metrics_binary(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]
    preds = np.argmax(logits, axis=1)
    out = {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "f1": f1_score(labels, preds, zero_division=0),
    }
    try:
        out["roc_auc"] = roc_auc_score(labels, probs)
    except Exception:
        out["roc_auc"] = np.nan
    return out

def train_bertweet_once(
    csv_path: str,
    text_col: str,
    label_col: str,
    model_name: str = "vinai/bertweet-base",
    max_len: int = 128,
    epochs: int = 3,
    train_bs: int = 16,
    eval_bs: int = 32,
    lr: float = 2e-5,
    balance_train: bool = True,
    balance_mode: str = "upsample",
    output_dir: str = "/content/bertweet_runs"
):
    set_seed(RANDOM_STATE)
    os.makedirs(output_dir, exist_ok=True)

    df = pd.read_csv(csv_path)
    df = clean_binary_df(df, text_col, label_col)

    train_val, test_df = train_test_split(
        df, test_size=0.2, random_state=RANDOM_STATE, stratify=df[label_col]
    )
    train_df, val_df = train_test_split(
        train_val, test_size=0.2, random_state=RANDOM_STATE, stratify=train_val[label_col]
    )

    if balance_train:
        train_df = balance_train_df(train_df, label_col, balance_mode)

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        ignore_mismatched_sizes=True
    )

    def tok(batch):
        return tokenizer(batch[text_col], truncation=True, max_length=max_len)

    ds_train = Dataset.from_pandas(train_df[[text_col, label_col]].rename(columns={label_col:"labels"}))
    ds_val = Dataset.from_pandas(val_df[[text_col, label_col]].rename(columns={label_col:"labels"}))
    ds_test = Dataset.from_pandas(test_df[[text_col, label_col]].rename(columns={label_col:"labels"}))

    ds_train = ds_train.map(tok, batched=True)
    ds_val = ds_val.map(tok, batched=True)
    ds_test = ds_test.map(tok, batched=True)

    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=lr,
        per_device_train_batch_size=train_bs,
        per_device_eval_batch_size=eval_bs,
        num_train_epochs=epochs,
        weight_decay=0.01,
        warmup_steps=50,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        report_to="none",
        save_total_limit=2
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=ds_train,
        eval_dataset=ds_val,
        data_collator=collator,
        compute_metrics=compute_metrics_binary,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()

    val_pred = trainer.predict(ds_val)
    test_pred = trainer.predict(ds_test)

    def build_report(pred_obj, split_name):
        logits = pred_obj.predictions
        probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]
        preds = np.argmax(logits, axis=1)
        labels = pred_obj.label_ids
        metrics = {
            "split": split_name,
            "accuracy": accuracy_score(labels, preds),
            "precision": precision_score(labels, preds, zero_division=0),
            "recall": recall_score(labels, preds, zero_division=0),
            "f1": f1_score(labels, preds, zero_division=0),
            "roc_auc": roc_auc_score(labels, probs) if len(np.unique(labels)) > 1 else np.nan
        }
        cm = confusion_matrix(labels, preds)
        rep = classification_report(labels, preds, output_dict=True, zero_division=0)
        return metrics, cm, rep, probs, preds, labels

    val_metrics, val_cm, _, _, _, _ = build_report(val_pred, "val")
    test_metrics, test_cm, _, test_probs, test_preds, _ = build_report(test_pred, "test")

    log_hist = pd.DataFrame(trainer.state.log_history)

    fig_loss = go.Figure()
    if "loss" in log_hist.columns:
        tmp = log_hist[log_hist["loss"].notna()].copy()
        if not tmp.empty:
            fig_loss.add_trace(go.Scatter(x=tmp["epoch"], y=tmp["loss"], mode="lines+markers", name="train_loss"))
    if "eval_loss" in log_hist.columns:
        tmp = log_hist[log_hist["eval_loss"].notna()].copy()
        if not tmp.empty:
            fig_loss.add_trace(go.Scatter(x=tmp["epoch"], y=tmp["eval_loss"], mode="lines+markers", name="val_loss"))
    fig_loss.update_layout(template="plotly_white", title="Криві навчання", xaxis_title="Епоха", yaxis_title="Втрата", height=420)

    fig_cm = px.imshow(
        test_cm,
        x=["Прогноз 0", "Прогноз 1"],
        y=["Істина 0", "Істина 1"],
        text_auto=True,
        color_continuous_scale="Blues",
        title="Матриця помилок (тест)"
    )
    fig_cm.update_layout(template="plotly_white", height=420)

    metrics_df = pd.DataFrame([val_metrics, test_metrics])

    pred_test_df = test_df.copy().reset_index(drop=True)
    pred_test_df["prob_1"] = test_probs
    pred_test_df["pred"] = test_preds

    md = f"""
**Розмір train:** `{len(train_df)}`
**Розмір val:** `{len(val_df)}`
**Розмір test:** `{len(test_df)}`
**Модель:** `{model_name}`
"""

    APP_STATE["trained_model"] = trainer.model
    APP_STATE["trained_tokenizer"] = tokenizer

    return md, metrics_df, fig_loss, fig_cm, pred_test_df, log_hist




def build_wordcloud_figure(text: str, title: str):
    clean = normalize_for_topics(text)
    clean = re.sub(r"\b\d+\b", " ", clean)
    clean = re.sub(r"\s+", " ", clean).strip()

    fig = go.Figure()
    if len(clean) < 3:
        fig.update_layout(template="plotly_white", title=title, height=320)
        return fig

    wc = WordCloud(width=900, height=420, background_color="white", collocations=False).generate(clean)
    fig = px.imshow(wc.to_array())
    fig.update_xaxes(showticklabels=False)
    fig.update_yaxes(showticklabels=False)
    fig.update_layout(template="plotly_white", title=title, coloraxis_showscale=False, height=320)
    return fig

def build_segment_share_pies(by_segment: pd.DataFrame):
    n = len(by_segment)
    cols = 4 if n >= 4 else max(1, n)
    rows = int(np.ceil(n / cols))

    fig = make_subplots(
        rows=rows,
        cols=cols,
        specs=[[{"type":"domain"}] * cols for _ in range(rows)],
        subplot_titles=[f"{r['segment_name']}" for _, r in by_segment.iterrows()]
    )

    for idx, (_, row) in enumerate(by_segment.iterrows(), start=1):
        rr = (idx - 1) // cols + 1
        cc = (idx - 1) % cols + 1
        fig.add_trace(
            go.Pie(
                labels=["Без виражених ознак", "Цифрова втома"],
                values=[100 - row["local_index"], row["local_index"]],
                textinfo="percent",
                showlegend=(idx == 1),
                sort=False
            ),
            row=rr,
            col=cc
        )

    fig.update_layout(
        template="plotly_white",
        height=min(max(420, 220 * rows), 900),
        margin=dict(l=30, r=30, t=70, b=30),
        title="Структура повідомлень у сегментах"
    )
    return fig

def build_author_profile_from_state(
    tau: float = 0.50,
    gamma: float = 50.0,
    alpha: float = 0.80
):
    tau = 0.50 if tau is None else float(tau)
    gamma = 50.0 if gamma is None else float(gamma)
    alpha = 0.80 if alpha is None else float(alpha)

    seg_df = APP_STATE.get("segmentation_df")
    model = APP_STATE.get("trained_model")
    tokenizer = APP_STATE.get("trained_tokenizer")
    text_col = APP_STATE.get("text_col")
    user_col = APP_STATE.get("user_col")

    if seg_df is None or text_col is None or user_col is None:
        raise ValueError("Спочатку виконайте сегментацію повідомлень.")
    if model is None or tokenizer is None:
        raise ValueError("Спочатку навчіть класифікатор цифрової втоми.")

    work = seg_df.copy()
    work = work[work["cluster"] != -1].copy()
    if work.empty:
        raise ValueError("Після сегментації не залишилося ненульових сегментів.")

    texts = work[text_col].astype(str).fillna("").tolist()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    model.eval()

    probs_all = []
    for i in range(0, len(texts), 32):
        batch = texts[i:i+32]
        enc = tokenizer(batch, truncation=True, max_length=128, padding=True, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            logits = model(**enc).logits
            probs = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
        probs_all.extend(probs.tolist())

    work["fatigue_prob"] = probs_all
    work["fatigue_bin"] = (work["fatigue_prob"] > tau).astype(int)
    work["segment_id"] = work["cluster"]

    by_segment = (
        work.groupby([user_col, "segment_id", "segment_name", "keywords"], dropna=False)
        .agg(
            n_messages=(text_col, "size"),
            mean_probability=("fatigue_prob", "mean"),
            local_index=("fatigue_bin", lambda x: float(np.mean(x) * 100.0)),
        )
        .reset_index()
        .rename(columns={user_col: "author_id", "keywords": "target_objects"})
        .sort_values("local_index", ascending=False)
        .reset_index(drop=True)
    )

    by_segment = by_segment.head(15).copy()
    by_segment["target_objects"] = by_segment["target_objects"].fillna("").map(clean_keywords_string)
    by_segment["critical"] = by_segment["local_index"] > gamma
    by_segment["segment_status"] = np.where(by_segment["critical"], "критичний", "некритичний")

    k = int(len(by_segment))
    m = int(by_segment["critical"].sum())
    coverage = float(m / k) if k > 0 else 0.0

    if coverage >= alpha:
        state_label = "Systemic Exhaustion"
        state_uk = "Системне цифрове виснаження"
    elif m > 0:
        state_label = "Situational Fatigue"
        state_uk = "Ситуативна цифрова втома"
    else:
        state_label = "Normal"
        state_uk = "Норма"

    author_id = by_segment["author_id"].iloc[0] if len(by_segment) else "невідомо"

    summary_md = f"""
## Інтегральний профіль цифрового виснаження

**Автор:** `{author_id}`
**Стан:** **{state_uk}** (`{state_label}`)
**CoverageΘ:** **{coverage:.3f}**
**Критичних сегментів:** **{m} із {k}**
**Пороги:** τ = `{tau:.2f}`, γ = `{gamma:.1f}`, α = `{alpha:.2f}`

Локальний індекс `eᵢ` = частка повідомлень сегмента, для яких оцінка моделі перевищує `τ`.
Інтегральний профіль автора формується через `CoverageΘ = count(eᵢ > γ) / k`.
"""

    fig_gauge = go.Figure(go.Indicator(
        mode="gauge+number",
        value=coverage * 100.0,
        number={"suffix":"%"},
        title={"text":"Охоплення критичних сегментів (CoverageΘ)"},
        gauge={
            "axis":{"range":[0,100]},
            "threshold":{"line":{"color":"red","width":3},"value":alpha*100.0},
            "steps":[
                {"range":[0, alpha*100.0], "color":"#d8ead3"},
                {"range":[alpha*100.0, 100], "color":"#f4cccc"}
            ],
            "bar":{"color":"#4c6ef5"}
        }
    ))
    fig_gauge.update_layout(template="plotly_white", height=330, margin=dict(l=20, r=20, t=50, b=20))

    fig_segments = px.bar(
        by_segment.sort_values("local_index", ascending=True),
        x="local_index",
        y="segment_name",
        orientation="h",
        color="segment_status",
        hover_data=["n_messages","mean_probability","target_objects"],
        title="Локальні індекси цифрової втоми за сегментами"
    )
    fig_segments.add_vline(x=gamma, line_width=2, line_dash="dash", line_color="red")
    fig_segments.update_layout(
        template="plotly_white",
        height=520,
        margin=dict(l=40, r=20, t=60, b=40),
        xaxis_title="Локальний індекс eᵢ, %",
        yaxis_title="Іменований сегмент"
    )

    fig_share = build_segment_share_pies(by_segment)

    top = by_segment.sort_values("local_index", ascending=False).head(3)
    wc_figs = []
    for _, row in top.iterrows():
        segid = row["segment_id"]
        txt = " ".join(work.loc[work["segment_id"] == segid, text_col].astype(str).tolist())
        wc_figs.append(build_wordcloud_figure(txt, f"Хмара слів: {row['segment_name']}"))
    while len(wc_figs) < 3:
        tmp = go.Figure()
        tmp.update_layout(template="plotly_white", height=320)
        wc_figs.append(tmp)

    out_table = by_segment[["segment_name","target_objects","n_messages","mean_probability","local_index","segment_status"]].copy()
    out_table.columns = [
        "Іменований сегмент",
        "Цільові об'єкти / ключові терми",
        "К-сть повідомлень",
        "Сер. ймовірність",
        "Локальний індекс eᵢ, %",
        "Статус"
    ]

    APP_STATE["evaluated_segments"] = by_segment.copy()
    return summary_md, fig_gauge, fig_segments, out_table, fig_share, wc_figs[0], wc_figs[1], wc_figs[2]




def ui_train(csv_file, text_col, label_col, model_name, max_len, epochs, train_bs, eval_bs, lr, balance_train, balance_mode):
    csv_path = csv_file.name if hasattr(csv_file, "name") else csv_file
    return train_bertweet_once(
        csv_path=csv_path,
        text_col=text_col,
        label_col=label_col,
        model_name=model_name,
        max_len=int(max_len),
        epochs=int(epochs),
        train_bs=int(train_bs),
        eval_bs=int(eval_bs),
        lr=float(lr),
        balance_train=bool(balance_train),
        balance_mode=balance_mode
    )

def ui_segmentation(csv_file, text_col, user_col, target_user, model_name, min_cluster_size, min_samples, top_n_keywords, lang):
    csv_path = csv_file.name if hasattr(csv_file, "name") else csv_file
    return run_segmentation(
        csv_path=csv_path,
        text_col=text_col,
        user_col=user_col,
        target_user=target_user,
        model_name=model_name,
        min_cluster_size=int(min_cluster_size),
        min_samples=int(min_samples),
        top_n_keywords=int(top_n_keywords),
        lang=lang
    )

def ui_profile(tau, gamma, alpha):
    tau = 0.50 if tau is None or tau == "" else float(tau)
    gamma = 50.0 if gamma is None or gamma == "" else float(gamma)
    alpha = 0.80 if alpha is None or alpha == "" else float(alpha)
    return build_author_profile_from_state(tau=tau, gamma=gamma, alpha=alpha)




with gr.Blocks(title="Інтерфейс експериментів для дисертації", theme=gr.themes.Soft()) as app:
    gr.Markdown("""
    # Інтерфейс експериментів для дисертації
    **Логіка:** навчання класифікатора цифрової втоми → визначення комунікативних сегментів → інтегральний профіль цифрового виснаження автора.
    """)

    with gr.Tab("1. Навчання нейромережі цифрової втоми"):
        with gr.Row():
            csv_train = gr.File(label="CSV-файл для навчання", file_types=[".csv"])
            with gr.Column():
                train_text_col = gr.Textbox(value="Tweet Text", label="Колонка з текстом")
                train_label_col = gr.Textbox(value="burnout", label="Колонка з міткою 0/1")
                train_model_name = gr.Textbox(value="vinai/bertweet-base", label="Модель")
                max_len = gr.Slider(32, 256, value=128, step=8, label="Максимальна довжина")
            with gr.Column():
                epochs = gr.Slider(1, 8, value=3, step=1, label="Кількість епох")
                train_bs = gr.Slider(4, 32, value=16, step=4, label="Train batch size")
                eval_bs = gr.Slider(4, 64, value=32, step=4, label="Eval batch size")
                lr = gr.Number(value=2e-5, label="Learning rate")
                balance_train = gr.Checkbox(value=True, label="Балансувати train")
                balance_mode = gr.Dropdown(["upsample", "downsample"], value="upsample", label="Режим балансування")

        btn_train = gr.Button("Навчити модель", variant="primary")
        train_md = gr.Markdown()
        train_metrics = gr.Dataframe(label="Метрики")
        train_loss_fig = gr.Plot(label="Криві навчання")
        train_cm_fig = gr.Plot(label="Матриця помилок")
        train_pred_df = gr.Dataframe(label="Передбачення на тесті")
        train_log_df = gr.Dataframe(label="Лог навчання")

        btn_train.click(
            ui_train,
            inputs=[csv_train, train_text_col, train_label_col, train_model_name, max_len, epochs, train_bs, eval_bs, lr, balance_train, balance_mode],
            outputs=[train_md, train_metrics, train_loss_fig, train_cm_fig, train_pred_df, train_log_df]
        )

    with gr.Tab("2. Сегментація та інтегральний профіль автора"):
        gr.Markdown("### 2.1. Сегментація та виявлення цільових об’єктів")
        with gr.Row():
            csv_seg = gr.File(label="CSV-файл для сегментації", file_types=[".csv"])
            with gr.Column():
                seg_text_col = gr.Textbox(value="post_text", label="Колонка з текстом")
                seg_user_col = gr.Textbox(value="user_id", label="Колонка з автором")
                seg_target_user = gr.Textbox(value="__AUTO__", label="ID автора або __AUTO__")
                seg_model_name = gr.Textbox(value="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", label="SentenceTransformer")
                seg_lang = gr.Dropdown(["uk", "en"], value="uk", label="Мова стоп-слів")
            with gr.Column():
                min_cluster_size = gr.Slider(5, 100, value=15, step=1, label="min_cluster_size")
                min_samples = gr.Slider(1, 30, value=5, step=1, label="min_samples")
                top_n_keywords = gr.Slider(5, 10, value=6, step=1, label="Кількість ключових термів")

        btn_seg = gr.Button("Запустити сегментацію", variant="primary")
        seg_md = gr.Markdown()
        seg_umap_fig = gr.Plot(label="UMAP-схема сегментів")
        seg_dist_fig = gr.Plot(label="Теплова карта відстаней")
        seg_summary_df = gr.Dataframe(label="Іменовані сегменти (до 15)")
        seg_examples_df = gr.Dataframe(label="Приклади повідомлень")

        btn_seg.click(
            ui_segmentation,
            inputs=[csv_seg, seg_text_col, seg_user_col, seg_target_user, seg_model_name, min_cluster_size, min_samples, top_n_keywords, seg_lang],
            outputs=[seg_md, seg_umap_fig, seg_dist_fig, seg_summary_df, seg_examples_df]
        )

        gr.Markdown("### 2.2. Інтегральний профіль цифрового виснаження")
        with gr.Row():
            tau = gr.Number(value=0.50, label="Поріг класифікації τ")
            gamma = gr.Number(value=50.0, label="Критичний поріг сегмента γ, %")
            alpha = gr.Number(value=0.80, label="Поріг системного поширення α")

        btn_profile = gr.Button("Побудувати інтегральний профіль", variant="primary")
        profile_md = gr.Markdown()
        profile_gauge = gr.Plot(label="CoverageΘ")
        profile_segments = gr.Plot(label="Локальні індекси сегментів")
        profile_table = gr.Dataframe(label="Таблиця сегментного профілю")
        profile_pies = gr.Plot(label="Структура повідомлень у сегментах")
        wc1 = gr.Plot(label="Хмара слів 1")
        wc2 = gr.Plot(label="Хмара слів 2")
        wc3 = gr.Plot(label="Хмара слів 3")

        btn_profile.click(
            ui_profile,
            inputs=[tau, gamma, alpha],
            outputs=[profile_md, profile_gauge, profile_segments, profile_table, profile_pies, wc1, wc2, wc3]
        )

app.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ea3797bb01b3c0cfab.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initi

Map:   0%|          | 0/1202 [00:00<?, ? examples/s]

Map:   0%|          | 0/301 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
